# Linear Blended Normalization

Blends Reinhard- and Macenko-normalized versions of the same images via a
weighted linear combination, controlled by `alpha`:

```
Blended_image = alpha * reinhard_image + (1 - alpha) * macenko_image
```

This produces a normalized dataset that combines properties of both stain
normalization methods. A single core function (`blend_images`) is used
throughout — only the folder structure being walked differs across the
usage examples below.

Update all `<PLACEHOLDER>` paths before running.


## Setup

In [ ]:
import os
import numpy as np
import cv2

## Core Blending Function

Blends two pre-normalized images (Reinhard and Macenko) of the same file
and saves the result.

In [ ]:
def blend_images(reinhard_img_path, macenko_img_path, output_img_path, alpha=0.5):
    """
    Blends a Reinhard-normalized and a Macenko-normalized image via a
    weighted linear combination and saves the result.

    Parameters
    ----------
    reinhard_img_path : str
        Path to the Reinhard-normalized image.
    macenko_img_path : str
        Path to the Macenko-normalized image (same filename/content as above).
    output_img_path : str
        Path to save the blended output image.
    alpha : float
        Weight given to the Reinhard image (1 - alpha is given to Macenko).
    """
    reinhard_img = cv2.imread(reinhard_img_path)
    macenko_img = cv2.imread(macenko_img_path)

    if reinhard_img is None or macenko_img is None:
        print(f"Error loading images: {output_img_path}. Skipping.")
        return

    assert reinhard_img.shape == macenko_img.shape, \
        f"Image dimensions do not match for {output_img_path}!"

    mixed_img = alpha * reinhard_img + (1 - alpha) * macenko_img
    mixed_img = np.clip(mixed_img, 0, 255).astype(np.uint8)

    os.makedirs(os.path.dirname(output_img_path), exist_ok=True)
    cv2.imwrite(output_img_path, mixed_img)

## Usage 1 — Multiclass Dataset (Class → Patient → Images)

For datasets organized as `class/patient_id/image.png`.

In [ ]:
def blend_multiclass_dataset(reinhard_base_path, macenko_base_path, mixed_base_path,
                              classes, alpha=0.5):
    for class_name in classes:
        reinhard_class_path = os.path.join(reinhard_base_path, class_name)
        macenko_class_path = os.path.join(macenko_base_path, class_name)
        mixed_class_path = os.path.join(mixed_base_path, class_name)

        patient_folders = os.listdir(reinhard_class_path)

        for patient_folder in patient_folders:
            reinhard_patient_path = os.path.join(reinhard_class_path, patient_folder)
            macenko_patient_path = os.path.join(macenko_class_path, patient_folder)
            mixed_patient_path = os.path.join(mixed_class_path, patient_folder)

            for file_name in os.listdir(reinhard_patient_path):
                blend_images(
                    os.path.join(reinhard_patient_path, file_name),
                    os.path.join(macenko_patient_path, file_name),
                    os.path.join(mixed_patient_path, file_name),
                    alpha=alpha,
                )
                print(f"Processed {file_name} in {patient_folder} for class {class_name}.")

    print("Mixing of multiclass dataset completed.")

In [ ]:
reinhard_base_path = "<REINHARD_BASE_PATH>"
macenko_base_path = "<MACENKO_BASE_PATH>"
mixed_base_path = "<MIXED_OUTPUT_PATH>"
classes = ["Astro", "Gbm", "Oligo"]  # update class names as needed

blend_multiclass_dataset(reinhard_base_path, macenko_base_path, mixed_base_path,
                          classes, alpha=0.5)

## Usage 2 — Single-Class Dataset (Patient → Images)

For datasets with only one class, organized as `patient_id/image.png`
(e.g. necrosis-only subset).

In [ ]:
def blend_single_class_dataset(reinhard_class_path, macenko_class_path, mixed_class_path,
                                alpha=0.5):
    patient_folders = os.listdir(reinhard_class_path)

    for patient_folder in patient_folders:
        reinhard_patient_path = os.path.join(reinhard_class_path, patient_folder)
        macenko_patient_path = os.path.join(macenko_class_path, patient_folder)
        mixed_patient_path = os.path.join(mixed_class_path, patient_folder)

        for file_name in os.listdir(reinhard_patient_path):
            blend_images(
                os.path.join(reinhard_patient_path, file_name),
                os.path.join(macenko_patient_path, file_name),
                os.path.join(mixed_patient_path, file_name),
                alpha=alpha,
            )
            print(f"Processed {file_name} in {patient_folder}.")

    print("Mixing of single-class dataset completed.")

In [ ]:
reinhard_class_path = "<REINHARD_CLASS_PATH>"   # e.g. .../Necrosis_Reinhard/GBM
macenko_class_path = "<MACENKO_CLASS_PATH>"     # e.g. .../Necrosis_Macenko/GBM
mixed_class_path = "<MIXED_CLASS_OUTPUT_PATH>"  # e.g. .../Necrosis_Mixed/GBM

blend_single_class_dataset(reinhard_class_path, macenko_class_path, mixed_class_path,
                            alpha=0.5)

## Usage 3 — Flat Folder (Images Only, No Class/Patient Subfolders)

For a flat folder of images with no nested class or patient structure
(e.g. a single binary-class patient folder).

In [ ]:
def blend_flat_folder(reinhard_folder, macenko_folder, mixed_folder, alpha=0.5):
    for file_name in os.listdir(reinhard_folder):
        blend_images(
            os.path.join(reinhard_folder, file_name),
            os.path.join(macenko_folder, file_name),
            os.path.join(mixed_folder, file_name),
            alpha=alpha,
        )
        print(f"Processed {file_name}.")

    print("Mixing of flat folder completed.")

In [ ]:
reinhard_folder = "<REINHARD_FOLDER_PATH>"
macenko_folder = "<MACENKO_FOLDER_PATH>"
mixed_folder = "<MIXED_OUTPUT_FOLDER_PATH>"

blend_flat_folder(reinhard_folder, macenko_folder, mixed_folder, alpha=0.5)